In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
block_size = 8
batch_size = 4
device = "mps" if torch.backends.mps.is_available() else "cpu"

In [2]:
with open ('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
print(chars)
vocab_size = len(chars)

['\n', ' ', '!', '&', '(', ')', ',', '-', '.', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '—', '‘', '’', '“', '”']


In [3]:
string_to_int = {ch:i for i,ch in enumerate(chars)}
int_to_string = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join(int_to_string[i] for i in l)

data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([41, 55, 52,  1, 44, 62, 61, 51, 52, 65, 53, 68, 59,  1, 44, 56, 73, 48,
        65, 51,  1, 62, 53,  1, 36, 73,  0,  0, 49, 72,  1, 33,  8,  1, 27, 65,
        48, 61, 58,  1, 23, 48, 68, 60,  0,  0,  0, 41, 55, 56, 66,  1, 49, 62,
        62, 58,  1, 56, 66,  1, 51, 52, 51, 56, 50, 48, 67, 52, 51,  1, 67, 62,
         1, 60, 72,  1, 54, 62, 62, 51,  1, 53, 65, 56, 52, 61, 51,  1,  3,  1,
        50, 62, 60, 65, 48, 51, 52,  0, 34, 72])


In [4]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    print('ix:', ix)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print('x:', x.shape)
print(x)
print('y:', y.shape)
print(y)

ix: tensor([ 75356, 105943, 154970,  70492])
x: torch.Size([4, 8])
tensor([[65, 48, 61,  1, 49, 72,  1, 55],
        [ 1, 48, 67,  1, 48, 59, 59,  0],
        [66,  1, 54, 62, 67,  1, 67, 70],
        [ 8,  0, 77, 30,  1, 48, 59, 70]], device='mps:0')
y: torch.Size([4, 8])
tensor([[48, 61,  1, 49, 72,  1, 55, 52],
        [48, 67,  1, 48, 59, 59,  0, 48],
        [ 1, 54, 62, 67,  1, 67, 70, 56],
        [ 0, 77, 30,  1, 48, 59, 70, 48]], device='mps:0')


In [5]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([41]) the target: 55
when input is tensor([41, 55]) the target: 52
when input is tensor([41, 55, 52]) the target: 1
when input is tensor([41, 55, 52,  1]) the target: 44
when input is tensor([41, 55, 52,  1, 44]) the target: 62
when input is tensor([41, 55, 52,  1, 44, 62]) the target: 61
when input is tensor([41, 55, 52,  1, 44, 62, 61]) the target: 51
when input is tensor([41, 55, 52,  1, 44, 62, 61, 51]) the target: 52


In [ ]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets):
        logits = self.token_embedding_table(idx)

        B, T, C = logits.shape
        logits = logits.view(B*T, C)
        targets = targets.view(B*T)
        loss = F.cross_entropy(logits, targets)
        
        return logits

SyntaxError: incomplete input (4010714244.py, line 3)